# 09 — M4 Audio Spectrogram Transformer (Akshat Yuvan)

**Model.** AST (Gong, Chung & Glass 2021, ref. 5): a ViT-style transformer
over 16×16 patches of a 128-bin log-mel filterbank, pre-trained on AudioSet
(`MIT/ast-finetuned-audioset-10-10-0.4593`, 0.4593 mAP). The 527-way
AudioSet head is replaced by one logit (real vs synthetic) and the whole
network is fine-tuned. M4 is the only pre-trained model of the four.

**Front-end: a deliberate, reported deviation from decision 3.** AST was
pre-trained on Kaldi-style fbank features (25 ms window, 10 ms hop, 128 bins,
1,024 frames, AudioSet mean/std), not on the shared librosa mel. Feeding it
the cached M2/M3 mels would throw away what pre-training learned, so its
features are computed here from the same raw audio, the same 10,697 clips
and the same 160,000-sample crop, with `ASTFeatureExtractor`. A 10 s clip
gives 998 frames; the extractor pads to 1,024 identically for both classes.

**Training: same protocol as M2/M3 in E1 except the learning rate.** Adam,
weight decay 1e-4, 1-epoch linear warmup then cosine to 0.01×, per step,
grad-clip 5, fp16 autocast + GradScaler (T4, decision 9), best checkpoint on
PLAIN validation accuracy (decision 10). Peak LR is 2e-5 rather than 1e-3:
1e-3 on a pre-trained transformer destroys the pre-trained weights in the
first few hundred steps; 1e-5 to 5e-5 is the standard fine-tuning range and
the one the AST paper uses. 4 epochs (pre-trained models converge in a few;
20 would mostly overfit and cost ~3 h). Batch 8 (memory).

**Attention interpretability.** Attention rollout (Abnar & Zuidema 2020) on
test clips: where in frequency does the [CLS]/[DIST] summary draw from?
Stated before the run: if M4 uses the same near-Nyquist cue E2 found in M1
and M2, rollout mass should be over-represented in the top patch row
(roughly 6–8 kHz).

In [ ]:
import os, glob, json, time, math, copy
import numpy as np, pandas as pd, torch, torch.nn as nn
import transformers
from transformers import ASTFeatureExtractor, ASTForAudioClassification

print(torch.__version__, transformers.__version__, '|',
      torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NO GPU')
assert torch.cuda.is_available()

def find_input(name, marker):
    d = f'/kaggle/input/datasets/akshatyuvan/{name}'
    if os.path.exists(f'{d}/{marker}'):
        return d
    hits = glob.glob(f'/kaggle/input/**/{name}/**/{marker}', recursive=True)
    assert hits, f'{name}/{marker} not found'
    return os.path.dirname(hits[0])

MELS_DIR = find_input('fmc-m2-mels', 'subset.csv')
RAW_DIR  = find_input('fakemusiccaps-raw', 'mustango')
REAL_DIR = find_input('musiccaps-real-16k', '-0Gj8-vB1q4.wav')
OUT, SEED = '/kaggle/working', 42
CKPT_NAME = 'MIT/ast-finetuned-audioset-10-10-0.4593'

sub    = pd.read_csv(f'{MELS_DIR}/subset.csv', dtype={'ytid': str})
labels = np.load(f'{MELS_DIR}/labels.npy')
assert (sub.label.values == labels).all() and len(sub) == 10697
# Raw-audio paths, remapped exactly as the M2 notebook did.
sub['path'] = (sub.path
    .str.replace('/kaggle/working/real', REAL_DIR, regex=False)
    .str.replace('/kaggle/input/datasets/akshatyuvan/fakemusiccaps-raw', RAW_DIR, regex=False))
assert sub.path.map(os.path.exists).all()
idx = {s: np.where((sub.split == s).values)[0] for s in ['train', 'val', 'test']}
print({s: len(v) for s, v in idx.items()})

In [ ]:
# AST features for all 10,697 clips, held in RAM as fp16 (2.8 GB).
import soundfile as sf
from joblib import Parallel, delayed

N_SAMPLES = 160000
fe = ASTFeatureExtractor.from_pretrained(CKPT_NAME)
assert (fe.sampling_rate, fe.num_mel_bins, fe.max_length) == (16000, 128, 1024)
print('AudioSet normalisation mean/std:', fe.mean, fe.std)

def ast_feat(path):
    y, sr = sf.read(path, dtype='float32')
    assert sr == 16000
    y = y[:N_SAMPLES] if len(y) >= N_SAMPLES else np.pad(y, (0, N_SAMPLES - len(y)))
    return fe(y, sampling_rate=16000, return_tensors='np')['input_values'][0].astype(np.float16)

t0 = time.time()
FB = np.empty((len(sub), 1024, 128), dtype=np.float16)
for s in range(0, len(sub), 1000):
    FB[s:s+1000] = np.stack(Parallel(n_jobs=4)(delayed(ast_feat)(p) for p in sub.path.values[s:s+1000]))
print('features', FB.shape, f'{time.time()-t0:.0f}s', '| NaNs', int(np.isnan(FB[:200].astype(np.float32)).sum()))

In [ ]:
# Data, model, training loop.
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import (balanced_accuracy_score, precision_score, recall_score,
                             f1_score, roc_auc_score, roc_curve)

class FbDS(Dataset):
    def __init__(self, arr, rows, ys):
        self.arr, self.rows, self.ys = arr, rows, ys
    def __len__(self): return len(self.rows)
    def __getitem__(self, i):
        return (torch.from_numpy(self.arr[self.rows[i]].astype(np.float32)),
                torch.tensor(float(self.ys[i])))

def loader(arr, rows, ys, shuffle=False, gen=None, bs=8):
    return DataLoader(FbDS(arr, rows, ys), batch_size=bs, shuffle=shuffle, generator=gen,
                      num_workers=2, pin_memory=True)

def make_m4(attn='sdpa'):
    return ASTForAudioClassification.from_pretrained(
        CKPT_NAME, num_labels=1, ignore_mismatched_sizes=True, attn_implementation=attn)

def predict(model, dl):
    model.eval(); out = []
    with torch.no_grad():
        for xb, _ in dl:
            with torch.amp.autocast('cuda', dtype=torch.float16):
                out.append(torch.sigmoid(model(xb.cuda()).logits.squeeze(1).float()).cpu().numpy())
    return np.concatenate(out)

def metrics(true, score):
    """Line-for-line the M1/M2/M3 metric code."""
    pred = (score > 0.5).astype(int)
    fpr, tpr, _ = roc_curve(true, score)
    eer = float(fpr[np.nanargmin(np.abs((1 - tpr) - fpr))])
    return {"balanced_accuracy": float(balanced_accuracy_score(true, pred)),
            "precision": float(precision_score(true, pred, zero_division=0)),
            "recall":    float(recall_score(true, pred)),
            "f1":        float(f1_score(true, pred)),
            "auc":       float(roc_auc_score(true, score)),
            "eer":       eer}

EPOCHS, PEAK_LR, CLIP = 4, 2e-5, 5.0

def train_m4(model, dl_tr, dl_va, y_va, pos_weight, ckpt):
    steps = EPOCHS * len(dl_tr); warm = len(dl_tr)
    def lr_lambda(s):
        if s < warm: return 0.1 + 0.9 * s / warm
        return 0.01 + 0.99 * 0.5 * (1 + math.cos(math.pi * (s - warm) / max(1, steps - warm)))
    opt = torch.optim.Adam(model.parameters(), lr=PEAK_LR, weight_decay=1e-4)
    sched = torch.optim.lr_scheduler.LambdaLR(opt, lr_lambda)
    lossf = nn.BCEWithLogitsLoss(pos_weight=torch.tensor(pos_weight, device='cuda'))
    scaler = torch.amp.GradScaler('cuda')
    best, best_ep, best_state, hist, t0 = -1.0, -1, None, [], time.time()
    for ep in range(EPOCHS):
        model.train(); run, n = 0.0, 0
        for xb, yb in dl_tr:
            xb, yb = xb.cuda(non_blocking=True), yb.cuda(non_blocking=True)
            opt.zero_grad(set_to_none=True)
            with torch.amp.autocast('cuda', dtype=torch.float16):
                loss = lossf(model(xb).logits.squeeze(1), yb)
            scaler.scale(loss).backward()
            scaler.unscale_(opt)
            torch.nn.utils.clip_grad_norm_(model.parameters(), CLIP)
            scaler.step(opt); scaler.update(); sched.step()
            run += loss.item() * len(yb); n += len(yb)
        s_va = predict(model, dl_va)
        va = float(((s_va > 0.5).astype(int) == y_va).mean())
        hist.append({"epoch": ep, "train_loss": round(run / n, 4), "val_acc": round(va, 4),
                     "val_bal_acc": round(float(balanced_accuracy_score(y_va, (s_va > 0.5).astype(int))), 4),
                     "elapsed_s": round(time.time() - t0, 1)})
        print(f'  epoch {ep}  train_loss {run/n:.4f}  val_acc {va:.4f}  ({time.time()-t0:.0f}s)', flush=True)
        if va > best:
            best, best_ep = va, ep
            best_state = {k: v.detach().clone() for k, v in model.state_dict().items()}
            torch.save({k: v.half() for k, v in best_state.items()}, ckpt)   # fp16: 172 MB
    model.load_state_dict(best_state)
    return best, best_ep, time.time() - t0, hist

In [ ]:
# Train M4 on the standard split. The balanced subsample is ~1:1, so
# pos_weight is n_real/n_synth in train (~1.0), recorded rather than assumed.
torch.manual_seed(SEED); np.random.seed(SEED)
y_tr, y_va, y_te = labels[idx['train']], labels[idx['val']], labels[idx['test']]
pos_weight = float((y_tr == 0).sum() / (y_tr == 1).sum())
g = torch.Generator().manual_seed(SEED)
dl_tr = loader(FB, idx['train'], y_tr, shuffle=True, gen=g)
dl_va = loader(FB, idx['val'], y_va, bs=32)
dl_te = loader(FB, idx['test'], y_te, bs=32)

model = make_m4().cuda()
n_params = sum(p.numel() for p in model.parameters())
print('params', n_params, '| pos_weight', round(pos_weight, 4), '| steps/epoch', len(dl_tr))
best, best_ep, train_time, hist = train_m4(model, dl_tr, dl_va, y_va, pos_weight, f'{OUT}/m4_best_fp16.pt')

s_te = predict(model, dl_te)
m4 = metrics(y_te, s_te)
pred = (s_te > 0.5).astype(int); gen_te = sub.generator.values[idx['test']]
results = {"model": "M4_AST", "seed": SEED, "device": "cuda", "gpu": torch.cuda.get_device_name(0),
           "train_time_s": round(train_time, 1), "best_val_acc": round(best, 4), "best_epoch": best_ep,
           "selection_metric": "val_accuracy", "params": n_params,
           **{k: round(v, 4) for k, v in m4.items()}, "n_test": int(len(y_te)),
           "test_recall_per_generator": {gg: round(float(pred[gen_te == gg].mean()), 4)
                                         for gg in sorted(set(gen_te[y_te == 1]))},
           "test_specificity_real": round(float((pred[y_te == 0] == 0).mean()), 4),
           "config": {"pretrained": CKPT_NAME, "epochs": EPOCHS, "batch_size": 8, "optimizer": "Adam",
                      "peak_lr": PEAK_LR, "weight_decay": 1e-4,
                      "schedule": "linear warmup 1 epoch (0.1->1.0) + cosine to 0.01x, per step",
                      "grad_clip_norm": CLIP, "amp": "fp16 autocast + GradScaler",
                      "pos_weight": round(pos_weight, 4),
                      "front_end": "ASTFeatureExtractor: Kaldi fbank 128 bins, 25 ms / 10 ms, 1024 frames, "
                                   f"AudioSet norm mean {fe.mean} std {fe.std}",
                      "transformers_version": transformers.__version__},
           "history": hist}
json.dump(results, open(f'{OUT}/m4_results.json', 'w'), indent=2)
print(json.dumps({k: v for k, v in results.items() if k != 'history'}, indent=2))

In [ ]:
# Comparison with M1-M3 (their saved numbers) and the training curve.
import matplotlib.pyplot as plt
fig, ax = plt.subplots(1, 2, figsize=(10, 3.5))
ax[0].plot([h['train_loss'] for h in hist], 'o-'); ax[0].set_title('M4 train loss'); ax[0].set_xlabel('epoch')
ax[1].plot([h['val_acc'] for h in hist], 'o-'); ax[1].set_title('M4 val accuracy'); ax[1].set_xlabel('epoch')
for a in ax: a.grid(alpha=.3)
plt.tight_layout(); plt.savefig(f'{OUT}/m4_training_curve.png', dpi=150); plt.show()

In [ ]:
# Attention rollout on test clips. The eager attention implementation is
# needed to return attention weights; the trained weights are copied in.
rm = make_m4('eager').cuda().eval()
rm.load_state_dict(model.state_dict())
F_DIM, T_DIM = rm.audio_spectrogram_transformer.embeddings.get_shape(rm.config)
assert (F_DIM, T_DIM) == (12, 101)

def rollout(x):
    """Abnar & Zuidema (2020): average heads, add the residual (0.5 I),
    renormalise rows, multiply through the 12 layers. AST's classifier reads
    the mean of the [CLS] and [DIST] tokens, so the rollout is taken from both
    and averaged. Returns a (12, 101) frequency x time map over patches."""
    with torch.no_grad():
        att = rm(x.cuda(), output_attentions=True).attentions
    R = None
    for A in att:
        A = A[0].float().mean(0)
        A = 0.5 * A + 0.5 * torch.eye(A.shape[-1], device=A.device)
        A = A / A.sum(-1, keepdim=True)
        R = A if R is None else A @ R
    r = 0.5 * (R[0, 2:] + R[1, 2:])
    return (r / r.sum()).reshape(F_DIM, T_DIM).cpu().numpy()

# 40 correctly classified clips per class / generator: 40 real + 40 x 5 synth.
rng = np.random.default_rng(SEED)
correct = pred == y_te
chosen = []
for gg in ['real'] + sorted(set(gen_te[y_te == 1])):
    pool = np.where((gen_te == gg) & correct)[0]
    chosen += list(rng.choice(pool, size=min(40, len(pool)), replace=False))
maps = np.stack([rollout(torch.from_numpy(FB[idx['test'][j]].astype(np.float32))[None]) for j in chosen])
gch = gen_te[chosen]

# Patch row r covers fbank bins 10r .. 10r+15. Kaldi mel scale, 20 Hz to
# Nyquist, 128 triangular filters over 130 equally spaced mel points.
mel = lambda f: 1127 * np.log(1 + f / 700); imel = lambda m: 700 * (np.exp(m / 1127) - 1)
pts = imel(np.linspace(mel(20), mel(8000), 130))
row_hz = [[round(float(pts[10 * r])), round(float(pts[10 * r + 17]))] for r in range(F_DIM)]
fprof = maps.sum(2)                               # (N, 12): share per frequency row
groups = {gg: gch == gg for gg in ['real'] + sorted(set(gch) - {'real'})}
att_res = {"method": "attention rollout (Abnar & Zuidema 2020), mean of [CLS] and [DIST] rows, "
                     "heads averaged, 0.5 residual", "n_per_group": 40, "row_hz": row_hz,
           "uniform_share_top_row": round(1 / F_DIM, 4),
           "groups": {gg: {"share_top_row": round(float(fprof[m, -1].mean()), 4),
                           "share_top_two_rows": round(float(fprof[m, -2:].sum(1).mean()), 4),
                           "profile": [round(float(v), 4) for v in fprof[m].mean(0)]}
                      for gg, m in groups.items()}}
json.dump(att_res, open(f'{OUT}/m4_attention_rollout.json', 'w'), indent=2)
for gg, v in att_res['groups'].items():
    print(f'{gg:18s} top row ({row_hz[-1][0]}-{row_hz[-1][1]} Hz) {v["share_top_row"]:.3f} '
          f'(uniform {1/F_DIM:.3f}) | top two rows {v["share_top_two_rows"]:.3f}')

fig, axes = plt.subplots(1, 2, figsize=(12, 3.8), gridspec_kw={'width_ratios': [1, 2.2]})
for gg, m in groups.items():
    axes[0].plot(fprof[m].mean(0), range(F_DIM), 'o-' if gg == 'real' else '.-',
                 color='k' if gg == 'real' else None, lw=2.5 if gg == 'real' else 1.2, label=gg)
axes[0].axvline(1 / F_DIM, color='grey', ls='--', lw=.8, label='uniform')
axes[0].set_yticks(range(F_DIM)); axes[0].set_yticklabels([f'{a/1000:.1f}' for a, _ in row_hz], fontsize=7)
axes[0].set_ylabel('patch row, lower edge (kHz)'); axes[0].set_xlabel('share of rollout mass')
axes[0].legend(fontsize=6); axes[0].grid(alpha=.3)
im = axes[1].imshow(maps[gch != 'real'].mean(0), origin='lower', aspect='auto', cmap='magma',
                    extent=[0, 10.24, 0, F_DIM])
axes[1].set_title('mean rollout, synthetic clips'); axes[1].set_xlabel('time (s)'); axes[1].set_ylabel('patch row')
plt.colorbar(im, ax=axes[1]); plt.tight_layout()
plt.savefig(f'{OUT}/m4_attention_rollout.png', dpi=150); plt.show()
print(sorted(os.listdir(OUT)))